# Academic Performance Prediction (G3)

This notebook trains and compares three regression models on the UCI Student Performance dataset (Math) to predict the final grade `G3` (0–20):

- **Linear Regression**: Interpretable baseline
- **Random Forest**: Non-linear model with hyperparameter tuning
- **K-Nearest Neighbors (KNN)**: Instance-based model with hyperparameter tuning

## Key decisions

- **Fixed seed** `random_state=42` at all random points (split, CV, models) to ensure reproducibility.
- **70 / 15 / 15 split** (train / validation / test). The validation set is folded back into training via `GridSearchCV` with 5-fold CV; the test set remains completely isolated until the final evaluation.
- **Explicit data leakage check with G1 and G2**: partial grades are strongly correlated with G3 (see heatmap in `data_processing.ipynb`) and including them can be a source of leakage. We train all models under two scenarios:
  - **Scenario A**: includes G1 and G2 (serves as an upper bound on performance, but of little practical use)
  - **Scenario B**: excludes G1 and G2 (answers the true research question: which socioeconomic, family, and study-habit factors predict performance)
- **Scaling** with `StandardScaler` inside a `Pipeline` for Linear Regression and KNN (required for KNN, beneficial for LR stability). Random Forest does not require scaling.

Dataset: **Mathematics** (395 students). The pipeline is directly reusable on `portuguese_processed.csv`.


In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42
os.makedirs('models', exist_ok=True)
print('Libraries loaded. random_state =', RANDOM_STATE)

## Load preprocessed data

We load `math_processed.csv` generated by `preprocessing.ipynb`. The file already contains binary encoding, one-hot variables, and the `pass` column. Here we only use the continuous target `G3`.


In [2]:
df = pd.read_csv('./data/math_processed.csv')
print('Shape:', df.shape)
df.head(3)


Shape: (395, 43)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,traveltime,studytime,...,Fjob_health,Fjob_other,Fjob_services,Fjob_teacher,reason_home,reason_other,reason_reputation,guardian_mother,guardian_other,pass
0,0,0,18,1,1,0,4,4,2,2,...,False,False,False,True,False,False,False,True,False,0
1,0,0,17,1,1,1,1,1,1,2,...,False,True,False,False,False,False,False,False,False,0
2,0,0,15,1,0,1,1,1,1,2,...,False,True,False,False,False,True,False,True,False,1


## Build scenarios A and B

We replicate the same feature definition as in `preprocessing.ipynb` (excluding targets `G3` and `pass`) and build two matrices:

- `X_A` includes G1 and G2: Contaminated with future information (partial grades that occur chronologically before G3 but are already very close to the target).
- `X_B` excludes G1 and G2: Free of leakage, based only on socioeconomic, family, and study-habit variables available before the course starts.


In [ ]:
TARGET = 'G3'
DROP_TARGETS = ['G3', 'pass']

features_A = [c for c in df.columns if c not in DROP_TARGETS]
features_B = [c for c in features_A if c not in ('G1', 'G2')]

X_A = df[features_A].astype(float)
X_B = df[features_B].astype(float)
y   = df[TARGET].astype(float)

print(f'Scenario A (with G1/G2): {X_A.shape[1]} features')
print(f'Scenario B (without G1/G2): {X_B.shape[1]} features')


Escenario A (con G1/G2): 41 features
Escenario B (sin G1/G2): 39 features


## 70 / 15 / 15 split

We perform a double split to obtain train (70%), validation (15%), and test (15%). We use the same seed and row order in both scenarios so the A vs B comparison is fair (the same students in each partition).

> Note: `GridSearchCV` will apply 5-fold CV on the train set. The validation set remains available for manual checks if needed, and the test set is only touched at the end.


In [ ]:
def split_70_15_15(X, y, random_state=RANDOM_STATE):
    # First split off test (15%), then divide the remainder into train (70/85) and val (15/85)
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y, test_size=0.15, random_state=random_state
    )
    val_ratio = 0.15 / 0.85
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval, test_size=val_ratio, random_state=random_state
    )
    return X_train, X_val, X_test, y_train, y_val, y_test

splits_A = split_70_15_15(X_A, y)
splits_B = split_70_15_15(X_B, y)

X_train_A, X_val_A, X_test_A, y_train_A, y_val_A, y_test_A = splits_A
X_train_B, X_val_B, X_test_B, y_train_B, y_val_B, y_test_B = splits_B

print(f'Train Scenario A: {X_train_A.shape}, val: {X_val_A.shape}, test: {X_test_A.shape}')
print(f'Train Scenario B: {X_train_B.shape}, val: {X_val_B.shape}, test: {X_test_B.shape}')


Escenario A — train: (275, 41), val: (60, 41), test: (60, 41)
Escenario B — train: (275, 39), val: (60, 39), test: (60, 39)


## Model definition and hyperparameter search

- **Linear Regression**: no hyperparameters to tune; we train directly with CV just to report its average performance.
- **Random Forest**: `GridSearchCV` over `n_estimators`, `max_depth`, `min_samples_split`.
- **KNN**: `GridSearchCV` over `n_neighbors` and `weights`.

The selection criterion is RMSE (lower is better) we use `neg_root_mean_squared_error` as `scoring`.


In [5]:
CV = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def build_models():
    lr_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model',  LinearRegression()),
    ])

    rf = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)
    rf_grid = {
        'n_estimators':      [100, 200],
        'max_depth':         [None, 10, 20],
        'min_samples_split': [2, 5],
    }

    knn_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model',  KNeighborsRegressor()),
    ])
    knn_grid = {
        'model__n_neighbors': [3, 5, 7, 9, 11],
        'model__weights':     ['uniform', 'distance'],
    }

    return lr_pipe, (rf, rf_grid), (knn_pipe, knn_grid)


## Training and evaluation

We train the three models on both scenarios. For LR we use `cross_val_score`; for RF and KNN we use `GridSearchCV` and keep the best estimator. We then evaluate once on the test set.


In [ ]:
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    mae  = float(mean_absolute_error(y_test, y_pred))
    r2   = float(r2_score(y_test, y_pred))
    return rmse, mae, r2

def train_scenario(X_train, y_train, X_test, y_test, tag):
    results = []
    fitted  = {}

    lr_pipe, (rf, rf_grid), (knn_pipe, knn_grid) = build_models()

    # --- Linear Regression ---
    lr_cv = -cross_val_score(lr_pipe, X_train, y_train,
                             scoring='neg_root_mean_squared_error', cv=CV, n_jobs=-1).mean()
    lr_pipe.fit(X_train, y_train)
    rmse, mae, r2 = evaluate(lr_pipe, X_test, y_test)
    results.append(('LinearRegression', tag, lr_cv, rmse, mae, r2, {}))
    fitted['LinearRegression'] = lr_pipe

    # --- Random Forest ---
    rf_gs = GridSearchCV(rf, rf_grid, scoring='neg_root_mean_squared_error',
                         cv=CV, n_jobs=-1, refit=True)
    rf_gs.fit(X_train, y_train)
    rf_cv = -rf_gs.best_score_
    rmse, mae, r2 = evaluate(rf_gs.best_estimator_, X_test, y_test)
    results.append(('RandomForest', tag, rf_cv, rmse, mae, r2, rf_gs.best_params_))
    fitted['RandomForest'] = rf_gs.best_estimator_

    # --- KNN ---
    knn_gs = GridSearchCV(knn_pipe, knn_grid, scoring='neg_root_mean_squared_error',
                          cv=CV, n_jobs=-1, refit=True)
    knn_gs.fit(X_train, y_train)
    knn_cv = -knn_gs.best_score_
    rmse, mae, r2 = evaluate(knn_gs.best_estimator_, X_test, y_test)
    results.append(('KNN', tag, knn_cv, rmse, mae, r2, knn_gs.best_params_))
    fitted['KNN'] = knn_gs.best_estimator_

    return results, fitted

print('Training Scenario A (with G1/G2)...')
results_A, models_A = train_scenario(X_train_A, y_train_A, X_test_A, y_test_A, 'A (with G1/G2)')

print('Training Scenario B (without G1/G2)...')
results_B, models_B = train_scenario(X_train_B, y_train_B, X_test_B, y_test_B, 'B (without G1/G2)')

print('Training completed.')


Entrenando Escenario A (con G1/G2)...
Entrenando Escenario B (sin G1/G2)...
Entrenamiento completado.


## Comparative results table

We consolidate the results of both scenarios into a single `DataFrame`. We report RMSE, MAE, and R² on the test set, along with the average RMSE obtained in 5-fold CV and the chosen hyperparameters.


In [ ]:
rows = results_A + results_B
results_df = pd.DataFrame(rows, columns=[
    'Model', 'Scenario', 'RMSE_CV', 'RMSE_test', 'MAE_test', 'R2_test', 'Best_params'
])
results_df = results_df.sort_values(['Scenario', 'RMSE_test']).reset_index(drop=True)

with pd.option_context('display.max_colwidth', None):
    display(results_df)

### Interpretation of results

- Scenario A (with G1 and G2) typically achieves very high R² (>0.8) because G1 and G2 are almost the same target. This confirms the leakage risk: the model is not learning to predict performance from early variables, but rather copying partial grades.
- Scenario B is the one that answers the true research question. R² values drop, which is expected, but the metrics are more honest and its explanatory factors (see `explainability.ipynb`) are the ones with practical value for the instructor and for a possible educational intervention.

For this reason, the explainability analysis will be done on Scenario B models.


## Model persistence

We save all trained models (both scenarios) with `joblib` in the `models/` folder so that `explainability.ipynb` can load them without retraining.


In [ ]:
for name, model in models_A.items():
    joblib.dump(model, f'models/{name}_scenarioA.joblib')
for name, model in models_B.items():
    joblib.dump(model, f'models/{name}_scenarioB.joblib')

joblib.dump({
    'features_A':  features_A,
    'features_B':  features_B,
    'X_train_B':   X_train_B,
    'X_test_B':    X_test_B,
    'y_train_B':   y_train_B,
    'y_test_B':    y_test_B,
    'X_train_A':   X_train_A,
    'X_test_A':    X_test_A,
    'y_train_A':   y_train_A,
    'y_test_A':    y_test_A,
}, 'models/data_splits.joblib')

results_df.to_csv('models/results_comparison.csv', index=False)
print('Models, splits and results table saved in models/')

## Notebook conclusion

1. 3 models x 2 scenarios = 6 configurations were trained, all with a fixed seed and 5-fold CV.
2. Scenario B (without G1/G2) is the one used in `explainability.ipynb` to answer the research question.
3. The models are serialized in `models/` ready to be loaded.
